# Credit Default Prediction — Give Me Some Credit
Predicting the probability a borrower defaults within 2 years, using the Kaggle "Give Me Some Credit" dataset.

**Pipeline:** load & clean data → handle missing values & outliers → train/test split → XGBoost with GridSearchCV (imbalance handled via `scale_pos_weight`) → evaluate with ROC/PR curves → cost-based threshold tuning → save model artifacts.

## 1. Imports

In [ ]:
import json
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

## 2. Load Data & Rename Columns

In [ ]:
df = pd.read_csv('cs-training.csv')

# Drop the unnamed index column that comes with this dataset
df = df.drop(columns=['Unnamed: 0'])

# Rename columns to simpler, friendlier names
df = df.rename(columns={
    'SeriousDlqin2yrs': 'default',
    'RevolvingUtilizationOfUnsecuredLines': 'credit_utilization',
    'NumberOfTime30-59DaysPastDueNotWorse': 'late_30_59_days',
    'DebtRatio': 'debt_ratio',
    'MonthlyIncome': 'monthly_income',
    'NumberOfOpenCreditLinesAndLoans': 'open_credit_lines',
    'NumberOfTimes90DaysLate': 'late_90_days',
    'NumberRealEstateLoansOrLines': 'real_estate_loans',
    'NumberOfTime60-89DaysPastDueNotWorse': 'late_60_89_days',
    'NumberOfDependents': 'dependents',
})

print(df.shape)
df.head()

## 3. Handle Missing Values
Impute with the median — robust to the outliers/skew present in `monthly_income` and `dependents`.

In [ ]:
df['monthly_income'] = df['monthly_income'].fillna(df['monthly_income'].median())
df['dependents'] = df['dependents'].fillna(df['dependents'].median())

df.isnull().sum()

## 4. Target Distribution
Confirms the class imbalance that drives the modeling choices below (`scale_pos_weight`, threshold tuning, PR-AUC over accuracy).

In [ ]:
sns.countplot(data=df, x='default')
plt.xticks([0, 1], labels=['No Default', 'Default'])
plt.title('Target Class Distribution')
plt.show()

default_rate = df['default'].value_counts()[1] / len(df['default'])
print(f"Default rate: {default_rate:.2%}")

## 5. Outlier / Data Quality Cleaning
This dataset has known data-quality issues:
- `credit_utilization` should be a 0–1 ratio but has extreme values
- `age` has invalid entries (e.g. 0)
- The three "late payment" columns use `96`/`98` as placeholder/error codes, not real counts

In [ ]:
df = df[(df['credit_utilization'] >= 0) & (df['credit_utilization'] <= 1)]
df = df[(df['age'] >= 18) & (df['age'] <= 95)]

for col in ['late_30_59_days', 'late_60_89_days', 'late_90_days']:
    df = df[~df[col].isin([96, 98])]

print(df.shape)

## 6. Train/Test Split

In [ ]:
X = df.drop(columns='default')
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.2%}  |  Test default rate: {y_test.mean():.2%}")

## 7. Model Training — XGBoost + GridSearchCV
Imbalance is handled via `scale_pos_weight` (computed from the training set only). No SMOTE — using both together over-corrects and tanks precision.

In [ ]:
ratio = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb = XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='auc',
    random_state=42,
)

param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [4, 5, 6],
    'learning_rate': [0.2, 0.3, 0.4],
    'subsample': [0.8, 0.9, 1.0],
}

cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=cv_stratified,
    n_jobs=-1,
    verbose=2,
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

best_xgb_model = grid_search.best_estimator_
predictions = best_xgb_model.predict(X_test)

## 8. Evaluation — Classification Report (default 0.5 threshold)

In [ ]:
print(classification_report(y_test, predictions))

## 9. ROC & Precision-Recall Curves
ROC-AUC summarizes overall separation; PR-AUC is the more honest metric given the ~7% positive class rate.

In [ ]:
y_proba = best_xgb_model.predict_proba(X_test)[:, 1]

# --- ROC curve ---
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc:.4f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Credit Default Prediction')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

print(f"ROC-AUC Score: {roc_auc:.4f}")

# --- Precision-Recall curve ---
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(7, 6))
plt.plot(recall, precision, label=f'XGBoost (PR-AUC = {pr_auc:.4f})', color='steelblue', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Credit Default Prediction')
plt.legend(loc='upper right')
plt.grid(alpha=0.3)
plt.show()

print(f"PR-AUC Score: {pr_auc:.4f}  (baseline / random-skill PR-AUC ~= {y_test.mean():.4f})")

## 10. Cost-Based Threshold Tuning
0.5 is not the right cutoff for a rare-event problem. Sweep thresholds and pick the one minimizing total business cost, given an assumed cost ratio between missed defaults (false negatives) and wrongly-rejected good borrowers (false positives).

In [ ]:
cost_false_negative = 9   # missed default - approved a loan that defaults (big loss)
cost_false_positive = 1   # rejected a good borrower - lost interest income (smaller loss)

thresholds = np.arange(0.01, 1.0, 0.01)
costs = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    total_cost = (fn * cost_false_negative) + (fp * cost_false_positive)
    costs.append(total_cost)

costs = np.array(costs)
best_threshold = thresholds[np.argmin(costs)]
min_cost = costs.min()

plt.figure(figsize=(8, 5))
plt.plot(thresholds, costs, color='crimson')
plt.axvline(best_threshold, linestyle='--', color='black',
            label=f'Optimal threshold = {best_threshold:.2f}')
plt.xlabel('Decision Threshold')
plt.ylabel('Total Business Cost')
plt.title('Cost-Based Threshold Tuning')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Optimal threshold (min cost): {best_threshold:.2f}")
print(f"Minimum total cost: {min_cost}")

y_pred_tuned = (y_proba >= best_threshold).astype(int)
print(f"\nClassification report at tuned threshold ({best_threshold:.2f}):")
print(classification_report(y_test, y_pred_tuned))

## 11. Save Model & Dependencies

In [ ]:
# Model (native format + joblib backup)
best_xgb_model.save_model('xgb_model.json')
joblib.dump(best_xgb_model, 'xgb_model.pkl')

# Feature order - required to reproduce predictions correctly
feature_names = list(X_train.columns)
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)

# Threshold + the cost assumptions and metrics behind it
model_metadata = {
    'best_threshold': float(best_threshold),
    'cost_false_negative': cost_false_negative,
    'cost_false_positive': cost_false_positive,
    'roc_auc': float(roc_auc),
    'pr_auc': float(pr_auc),
    'best_params': grid_search.best_params_,
    'scale_pos_weight': float(ratio),
}
with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=4)

# Manual cleaning constants applied to raw data - reapply identically to new data
preprocessing_params = {
    'credit_utilization_min': 0.0,
    'credit_utilization_max': 1.0,
    'age_min': 18,
    'age_max': 95,
    'late_payment_placeholder_codes': [96, 98],
    'monthly_income_impute': 'median',
    'dependents_impute': 'median',
}
with open('preprocessing_params.json', 'w') as f:
    json.dump(preprocessing_params, f, indent=4)

saved_files = ['xgb_model.json', 'xgb_model.pkl', 'feature_names.json', 'model_metadata.json', 'preprocessing_params.json']
print("All artifacts saved to the current directory:")
for file in saved_files:
    print(f"  - {file}")